# PE Ablation — Full Training Run (GPU Server)

All commands run on the server. Outputs are saved inline as proof of execution.

**Paths (server-local, no Drive mount needed)**
```
HI train : processed_data/tokenized/train_1m_hi_en.pt
HI val   : processed_data/tokenized/val_hi_en.pt
BN train : processed_data_bn/tokenized/train_bn_en.pt
BN val   : processed_data_bn/tokenized/val_bn_en.pt
```

In [1]:
import os, datetime, json, pathlib
os.chdir('/home/m240479cs/projects/Neur')

HI_TR = 'processed_data/tokenized/train_1m_hi_en.pt'
HI_VA = 'processed_data/tokenized/val_hi_en.pt'
BN_TR = 'processed_data_bn/tokenized/train_bn_en.pt'
BN_VA = 'processed_data_bn/tokenized/val_bn_en.pt'

COMMON = '--num-steps 75000 --eval-every 5000 --batch-size 256 --grad-accum 1 --max-seq-len 192 --seed 7'

print('Session started:', datetime.datetime.now())
print('Working dir:', os.getcwd())
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

Session started: 2026-04-16 09:42:05.279823
Working dir: /home/m240479cs/projects/Neur
NVIDIA A100 80GB PCIe, 81920 MiB, 575.57.08


In [2]:
# ── STATUS: show which runs already have checkpoints ─────────────────────────
import json, pathlib, datetime

logs = pathlib.Path('outputs/logs')
ckpts = pathlib.Path('outputs/checkpoints')

PE_TYPES = [ 'rope', 'asrope', 'asrope2', 'asrope3']
LANGS    = ['hi', 'bn']

print(f"{'Run':<20} {'Status':<12} {'Val Loss':>10} {'Best Step':>10} {'Saved At'}")
print('-' * 75)
for lang in LANGS:
    for pe in PE_TYPES:
        name = f'{pe}_{lang}'
        summary = logs / name / 'run_summary.json'
        best_pt = ckpts / name / 'best.pt'
        if summary.exists():
            d = json.loads(summary.read_text())
            mtime = datetime.datetime.fromtimestamp(best_pt.stat().st_mtime).strftime('%m-%d %H:%M') if best_pt.exists() else '—'
            print(f"{name:<20} {'DONE':<12} {d.get('best_val_loss',0):>10.4f} {d.get('best_step',0):>10} {mtime}")
        elif best_pt.exists():
            mtime = datetime.datetime.fromtimestamp(best_pt.stat().st_mtime).strftime('%m-%d %H:%M')
            metrics = logs / name / 'metrics.jsonl'
            steps = sum(1 for _ in open(metrics)) if metrics.exists() else 0
            print(f"{name:<20} {'PARTIAL':<12} {'—':>10} {steps:>10} {mtime}")
        else:
            print(f"{name:<20} {'MISSING':<12} {'—':>10} {'—':>10} —")

Run                  Status         Val Loss  Best Step Saved At
---------------------------------------------------------------------------
rope_hi              DONE             3.0413      75000 04-15 20:05
asrope_hi            DONE             3.0367      75000 04-15 21:22
asrope2_hi           DONE             3.0442      75000 04-15 23:09
asrope3_hi           DONE             3.0344      75000 04-16 00:41
rope_bn              DONE             3.3336      75000 04-16 02:28
asrope_bn            PARTIAL               —      71210 04-16 04:30
asrope2_bn           MISSING               —          — —
asrope3_bn           MISSING               —          — —


---
## Hindi — Hi→En Training
Run each cell individually. Output is saved permanently in this notebook.

In [ ]:
import datetime; print('START sinusoidal_hi:', datetime.datetime.now())
!python -m pipeline.step3_train \
    --pe-type sinusoidal \
    --tokenized-train processed_data/tokenized/train_1m_hi_en.pt \
    --tokenized-val   processed_data/tokenized/val_hi_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name sinusoidal_hi
import datetime; print('END sinusoidal_hi:', datetime.datetime.now())

In [ ]:
import datetime; print('START rope_hi:', datetime.datetime.now())
!python -m pipeline.step3_train \
    --pe-type rope \
    --tokenized-train processed_data/tokenized/train_1m_hi_en.pt \
    --tokenized-val   processed_data/tokenized/val_hi_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name rope_hi
import datetime; print('END rope_hi:', datetime.datetime.now())

In [ ]:
import datetime; print('START asrope_hi:', datetime.datetime.now())
!python -m pipeline.step3_train \
    --pe-type asrope \
    --tokenized-train processed_data/tokenized/train_1m_hi_en.pt \
    --tokenized-val   processed_data/tokenized/val_hi_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name asrope_hi
import datetime; print('END asrope_hi:', datetime.datetime.now())

In [ ]:
import datetime; print('START asrope2_hi:', datetime.datetime.now())
!python -m pipeline.step3_train \
    --pe-type asrope2 \
    --tokenized-train processed_data/tokenized/train_1m_hi_en.pt \
    --tokenized-val   processed_data/tokenized/val_hi_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name asrope2_hi
import datetime; print('END asrope2_hi:', datetime.datetime.now())

In [ ]:
import datetime; print('START asrope3_hi:', datetime.datetime.now())
!python -m pipeline.step3_train \
    --pe-type asrope3 \
    --tokenized-train processed_data/tokenized/train_1m_hi_en.pt \
    --tokenized-val   processed_data/tokenized/val_hi_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name asrope3_hi
import datetime; print('END asrope3_hi:', datetime.datetime.now())

In [ ]:
import datetime; print('START ccrope_hi:', datetime.datetime.now())
!python -m pipeline.step3_train \
    --pe-type ccrope \
    --tokenized-train processed_data/tokenized/train_1m_hi_en.pt \
    --tokenized-val   processed_data/tokenized/val_hi_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name ccrope_hi
import datetime; print('END ccrope_hi:', datetime.datetime.now())

In [ ]:
import datetime; print('START dsrope_hi:', datetime.datetime.now())
!python -m pipeline.step3_train \
    --pe-type dsrope \
    --tokenized-train processed_data/tokenized/train_1m_hi_en.pt \
    --tokenized-val   processed_data/tokenized/val_hi_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name dsrope_hi
import datetime; print('END dsrope_hi:', datetime.datetime.now())

---
## Bengali — Bn→En Training

In [3]:
# Quick validation: ensure notebook uses project venv Python with torch available
!/home/m240479cs/projects/Neur/.venv/bin/python -c "import torch; import sys; print(sys.executable); print(torch.__version__)"
!cd /home/m240479cs/projects/Neur && /home/m240479cs/projects/Neur/.venv/bin/python -m pipeline.step3_train --help | head -n 20

/home/m240479cs/projects/Neur/.venv/bin/python
2.5.1+cu121
usage: step3_train.py [-h] [--tokenizer TOKENIZER] [--max-seq-len MAX_SEQ_LEN]
                      [--d-model D_MODEL] [--n-layers N_LAYERS]
                      [--n-heads N_HEADS]
                      [--pe-type {sinusoidal,rope,asrope,asrope2,asrope3,ccrope,dsrope,none}]
                      [--batch-size BATCH_SIZE]
                      [--learning-rate LEARNING_RATE]
                      [--weight-decay WEIGHT_DECAY] [--num-steps NUM_STEPS]
                      [--eval-every EVAL_EVERY] [--grad-accum GRAD_ACCUM]
                      [--val-subset VAL_SUBSET] [--seed SEED] [--use-bf16]
                      [--no-use-bf16] [--use-compile] [--no-use-compile]
                      [--pin-memory] [--no-pin-memory]
                      [--dataloader-workers DATALOADER_WORKERS]
                      [--checkpoint-light] [--no-checkpoint-light]
                      [--tokenized-train TOKENIZED_TRAIN]
                  

In [ ]:
import datetime; print('START sinusoidal_bn:', datetime.datetime.now())
!python -m pipeline.step3_train \
    --pe-type sinusoidal \
    --tokenized-train processed_data_bn/tokenized/train_bn_en.pt \
    --tokenized-val   processed_data_bn/tokenized/val_bn_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name sinusoidal_bn
import datetime; print('END sinusoidal_bn:', datetime.datetime.now())

In [ ]:
import datetime; print('START rope_bn:', datetime.datetime.now())
!python -m pipeline.step3_train \
    --pe-type rope \
    --tokenized-train processed_data_bn/tokenized/train_bn_en.pt \
    --tokenized-val   processed_data_bn/tokenized/val_bn_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name rope_bn
import datetime; print('END rope_bn:', datetime.datetime.now())

In [ ]:
import datetime; print('START asrope_bn:', datetime.datetime.now())
!cd /home/m240479cs/projects/Neur && /home/m240479cs/projects/Neur/.venv/bin/python -m pipeline.step3_train \
    --pe-type asrope \
    --tokenized-train processed_data_bn/tokenized/train_bn_en.pt \
    --tokenized-val   processed_data_bn/tokenized/val_bn_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name asrope_bn
import datetime; print('END asrope_bn:', datetime.datetime.now())

START asrope_bn: 2026-04-16 09:43:19.700931
Traceback (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/m240479cs/projects/Neur/pipeline/step3_train.py", line 21, in <module>
    import torch
ModuleNotFoundError: No module named 'torch'
END asrope_bn: 2026-04-16 09:43:19.966523


In [ ]:
import datetime; print('START asrope2_bn:', datetime.datetime.now())
!cd /home/m240479cs/projects/Neur && /home/m240479cs/projects/Neur/.venv/bin/python -m pipeline.step3_train \
    --pe-type asrope2 \
    --tokenized-train processed_data_bn/tokenized/train_bn_en.pt \
    --tokenized-val   processed_data_bn/tokenized/val_bn_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name asrope2_bn
import datetime; print('END asrope2_bn:', datetime.datetime.now())

In [ ]:
import datetime; print('START asrope3_bn:', datetime.datetime.now())
!cd /home/m240479cs/projects/Neur && /home/m240479cs/projects/Neur/.venv/bin/python -m pipeline.step3_train \
    --pe-type asrope3 \
    --tokenized-train processed_data_bn/tokenized/train_bn_en.pt \
    --tokenized-val   processed_data_bn/tokenized/val_bn_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name asrope3_bn
import datetime; print('END asrope3_bn:', datetime.datetime.now())

In [ ]:
import datetime; print('START ccrope_bn:', datetime.datetime.now())
!python -m pipeline.step3_train \
    --pe-type ccrope \
    --tokenized-train processed_data_bn/tokenized/train_bn_en.pt \
    --tokenized-val   processed_data_bn/tokenized/val_bn_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name ccrope_bn
import datetime; print('END ccrope_bn:', datetime.datetime.now())

In [ ]:
import datetime; print('START dsrope_bn:', datetime.datetime.now())
!python -m pipeline.step3_train \
    --pe-type dsrope \
    --tokenized-train processed_data_bn/tokenized/train_bn_en.pt \
    --tokenized-val   processed_data_bn/tokenized/val_bn_en.pt \
    --num-steps 75000 --eval-every 5000 \
    --batch-size 256 --grad-accum 1 \
    --max-seq-len 192 --seed 7 \
    --run-name dsrope_bn
import datetime; print('END dsrope_bn:', datetime.datetime.now())

---
## Final Results Table
Run after all training cells complete.

In [ ]:
import json, pathlib, datetime

logs  = pathlib.Path('outputs/logs')
ckpts = pathlib.Path('outputs/checkpoints')

PE_TYPES = ['sinusoidal', 'rope', 'asrope', 'asrope2', 'asrope3', 'ccrope', 'dsrope']
LANGS    = ['hi', 'bn']

print(f"Generated: {datetime.datetime.now()}")
print()
print(f"{'PE Type':<12} {'Hi Val Loss':>12} {'Hi Step':>9} {'Bn Val Loss':>12} {'Bn Step':>9}")
print('-' * 58)
for pe in PE_TYPES:
    row = {'hi': None, 'bn': None}
    for lang in LANGS:
        f = logs / f'{pe}_{lang}' / 'run_summary.json'
        if f.exists():
            d = json.loads(f.read_text())
            row[lang] = d
    hi = row['hi']
    bn = row['bn']
    hi_loss = f"{hi['best_val_loss']:.4f}" if hi else 'MISSING'
    hi_step = f"{hi['best_step']}" if hi else '—'
    bn_loss = f"{bn['best_val_loss']:.4f}" if bn else 'MISSING'
    bn_step = f"{bn['best_step']}" if bn else '—'
    print(f"{pe:<12} {hi_loss:>12} {hi_step:>9} {bn_loss:>12} {bn_step:>9}")

print()
print('Checkpoint sizes:')
for lang in LANGS:
    for pe in PE_TYPES:
        p = ckpts / f'{pe}_{lang}' / 'best.pt'
        if p.exists():
            mtime = datetime.datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
            print(f"  {pe}_{lang}: {p.stat().st_size/1e6:.1f} MB  saved={mtime}")

---
## Skip already-done runs
These runs already completed — **do not re-run** unless you want to overwrite:
- `rope_hi` — saved Apr 15 20:05
- `asrope_hi` — saved Apr 15 21:22  
- `asrope2_hi` — saved Apr 15 23:09
- `asrope3_hi` — saved Apr 16 00:41
- `rope_bn` — saved Apr 16 02:28
- `asrope_bn` — saved Apr 16 04:30 (71,210/75,000 steps — usable)